# 1 Spark Sesssion

*Notebook 1 of 9*

In [ ]:
spark

# 2 Lazy Evaluation

*Notebook 2 of 9*

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Lazy Evaluation

In [ ]:
data = [
    (1,"lisa","lisa@random.com"),
    (2,"john","john@random.com"),
    (3,"mary","mary@random.com"),
    (4,"peter","peter@random.com"),
    (5,"jane","jane@random.com"),
    (6,"jim","jim@random.com"),
    (7,"sara","sara@random.com"),
    (8,"bob","bob@random.com"),
    (9,"alex","alex@random.com")
]

columns = ["id","name","email"]

df_new = spark.createDataFrame(data,columns)

# Transformation - 1
df_new = df_new.select("id","email")

# Transformation - 2
df_new = df_new.filter(col("id") > 5)

In [ ]:
display(df_new)

# 3 Spark Catalyst Optimizer

*Notebook 3 of 9*

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
data = [
    (1,"lisa","lisa@random.com"),
    (2,"john","john@random.com"),
    (3,"mary","mary@random.com"),
    (4,"peter","peter@random.com"),
    (5,"jane","jane@random.com"),
    (6,"jim","jim@random.com"),
    (7,"sara","sara@random.com"),
    (8,"bob","bob@random.com"),
    (9,"alex","alex@random.com")
]

columns = ["id","name","email"]

df_new = spark.createDataFrame(data,columns)

# Transformation - 1
df_new = df_new.select("id","email")

# Transformation - 2
df_new = df_new.filter(col("id") > 5)

In [ ]:
display(df_new)

In [ ]:
df_new.explain()

# 4 jobs stages tasks

*Notebook 4 of 9*

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
df = spark.read.format("csv")\
        .option("header",True)\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

In [ ]:
# Wide Transformation
df = df.repartition(2)

# Narrow Transformation - 1
df = df.select("order_id","customer_id")

# Narrow Transformation - 2
df = df.filter(col("order_id")==1001)

# Wide Transformation
df = df.groupBy("customer_id").count()

In [ ]:
display(df)

# 5 repartition coalesce

*Notebook 5 of 9*

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
df = spark.read.format("csv")\
        .option("header",True)\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/")

In [ ]:
display(df)

In [ ]:
display(df.withColumn("spark_partition",spark_partition_id())\
    .groupBy("spark_partition")\
    .agg(count("*").alias("count"))\
    .orderBy("spark_partition"))

In [ ]:
df_repart = df.repartition(4)
display(df_repart.withColumn("spark_partition",spark_partition_id())\
    .groupBy("spark_partition")\
    .agg(count("*").alias("count"))\
    .orderBy("spark_partition"))

In [ ]:
df_coal = df_repart.coalesce(2)
display(df_coal.withColumn("spark_partition",spark_partition_id())\
    .groupBy("spark_partition")\
    .agg(count("*").alias("count"))\
    .orderBy("spark_partition"))

# 6 spark query plans

*Notebook 6 of 9*

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
df = spark.read.format("csv")\
        .option("header",True)\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/")

In [ ]:
display(df)

## Query Plans

In [ ]:
# This will show all the query plans
df.explain(True)

In [ ]:
# This will only show you the physical plan
df.explain()

## Query Plan For Repartition

In [ ]:
df_repart = df.repartition(4)
df_repart.explain()

%md
## Query Plan For Coalesce

In [ ]:
df_coal = df.coalesce(2)
df_coal.explain()

## Query Plan For Aggregate

In [ ]:
df_grp = df.withColumn("spark_partition",spark_partition_id())\
    .groupBy("spark_partition")\
    .agg(count("*").alias("count"))\
    .orderBy("spark_partition")

df_grp.explain()

# 7 PySpark Guide

*Notebook 7 of 9*

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# DATA READING

### Reading CSVs

In [ ]:
df_csv = spark.read.format("csv")\
      .option("header",True)\
      .option("inferSchema",True)\
      .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

display(df_csv)

### Reading JSONs

In [ ]:
df_json = spark.read.format("JSON")\
        .option("inferSchema",True)\
        .option("multiLine",True)\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.json")

display(df_json)

### Reading Parquet

In [ ]:
df_parquet = spark.read.format("parquet")\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/part-00000-tid-orders.c000.snappy.parquet")

display(df_parquet)

### Reading JDBC

In [ ]:
# my_url = "jdbc:postgresql://localhost:5432/mydb"

# # Option - 1 To Create DF

# my_connection = {
#   "user": "postgres",
#   "password": "postgres",
#   "driver": "org.postgresql.Driver"
# }

# df = spark.read.jdbc(url = my_url, table = "orders", properties = my_connection)




# # Option - 2 To Create DF
# df = spark.read.format("jdbc")\
#           .option("url", my_url)\
#           .option("dbtable", "mydb.orders")\
#           .option("user", "postgres")\
#           .option("password", "postgres")\
#           .option("driver", "org.postgresql.Driver")\
#           .load()

# CORRUPT RECORDS MODES

### Permissive

In [ ]:
df_json = spark.read.format("JSON")\
        .option("inferSchema",True)\
        .option("multiLine",True)\
        .option("mode","PERMISSIVE")\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.json")

display(df_json)

### DROPMALFORMED

In [ ]:
df_json = spark.read.format("JSON")\
        .option("inferSchema",True)\
        .option("multiLine",True)\
        .option("mode","DROPMALFORMED")\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.json")

display(df_json)

### FAILFAST

In [ ]:
df_json = spark.read.format("JSON")\
        .option("inferSchema",True)\
        .option("multiLine",True)\
        .option("mode","FAILFAST")\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.json")

display(df_json)

%md
# Schema

### StructType

In [ ]:
df_csv.schema

In [ ]:
my_custom_schema = StructType([StructField('order_id', StringType(), True), StructField('customer_id', StringType(), True), StructField('order_date', DateType(), True), StructField('product_id', StringType(), True), StructField('quantity', IntegerType(), True), StructField('price', DoubleType(), True), StructField('order_status', StringType(), True), StructField('shipping_address', StringType(), True), StructField('city', StringType(), True), StructField('country', StringType(), True), StructField('payment_method', StringType(), True), StructField('discount', DoubleType(), True), StructField('category', StringType(), True), StructField('sales_rep', StringType(), True), StructField('region', StringType(), True), StructField('ship_date', DateType(), True), StructField('delivery_days', IntegerType(), True), StructField('returned', StringType(), True), StructField('gender', StringType(), True)])

df_csv_custom = spark.read.format("csv")\
      .option("header",True)\
      .schema(my_custom_schema)\
      .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

display(df_csv_custom)

### DDL Schema

In [ ]:
my_ddl_schema = """
  order_id INT,
  customer_id STRING,
  order_date DATE,
  product_id STRING,
  quantity INT,
  price DOUBLE,
  order_status STRING,
  shipping_address STRING,
  city STRING,
  country STRING,
  payment_method STRING,
  discount DOUBLE,
  category STRING,
  sales_rep STRING,
  region STRING,
  ship_date DATE,
  delivery_days INT,
  returned STRING,
  gender STRING
  """

df_csv_ddl = spark.read.format("csv")\
      .option("header",True)\
      .schema(my_ddl_schema)\
      .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

display(df_csv_ddl)

# SELECT

In [ ]:
df_select = df_csv.select('customer_id',col('city'),'country')
display(df_select)

# ALIAS

In [ ]:
df_alias = df_csv.select('customer_id',col('city').alias('customer_city'),col('country').alias('customer_country'))

display(df_alias)

# FILTER

### Scenario-1

In [ ]:
df_filter1 = df_csv.filter(col('order_status')=='Returned')
display(df_filter1)

### Scenario-2

In [ ]:
df_filter2 = df_csv.filter((col('order_status')=='Returned') | (col('order_status') == 'Cancelled'))
display(df_filter2)

### Scenario-3

In [ ]:
desired_order = ['Returned','Cancelled','Shipped']

df_filter3 = df_csv.filter(col('order_status').isin(desired_order))
display(df_filter3)

# withColumnRenamed

In [ ]:
df_renamed = df_csv.withColumnRenamed('order_status','order_status_info')
display(df_renamed)

# withColumn

### Scenario-1

In [ ]:
df_mod1 = df_csv.withColumn('flag',lit(0))
display(df_mod1)

### Scenario-2

In [ ]:
df_mod2 = df_csv.withColumn('shipping_address',regexp_replace('shipping_address',',.*',''))
display(df_mod2)

### Scenario-3

In [ ]:
df_mod3 = df_csv.withColumn("total_price",col('price')*col('quantity'))\
            .withColumn("total_price",round(col('total_price'),2))
display(df_mod3)

# TYPE CASTING

In [ ]:
df_cast1 = df_csv.withColumn("order_id",col('order_id').cast(StringType()))
display(df_cast1)

In [ ]:
df_cast2 = df_csv.withColumn("order_id",col('order_id').cast('STRING'))
display(df_cast2)

# SORTING

### Scenario-1

In [ ]:
df_sort1 = df_csv.sort(col('order_date').desc())
display(df_sort1)

### Scenario-2

In [ ]:
df_sort2 = df_csv.sort(['order_date','quantity'],ascending=[0,1])
display(df_sort2)

# LIMIT

In [ ]:
display(df_csv.limit(10))

# DROP

In [ ]:
df_drop = df_csv.drop('order_status','shipping_address')
display(df_drop)

# DROP DUPLICATES

### Scenario-1

In [ ]:
df_dedups = df_csv.dropDuplicates()
display(df_dedups)

### Scenario-2

In [ ]:
df_dedups2 = df_csv.dropDuplicates(subset=['order_date','product_id'])
display(df_dedups2)

# Union & UnionByName

### Union

In [ ]:
data1 = [(1,'aa','123'),(2,'bb','456'),(3,'cc','789')]
df1 = spark.createDataFrame(data1,['id','name','address'])
data2 = [(4,'aa','123'),(5,'bb','456'),(6,'cc','789'),(7,'dd','123')]
df2 = spark.createDataFrame(data2,['id','name','address'])

df_union = df1.union(df2)
display(df_union)

### UnionByName

In [ ]:
data1 = [(1,'aa','123'),(2,'bb','456'),(3,'cc','789')]
df1 = spark.createDataFrame(data1,['id','name','address'])
data2 = [('123','aa',4),('456','bb',5),('789','cc',6),('123','dd',7)]
df2 = spark.createDataFrame(data2,['address','name','id'])

df_union = df1.unionByName(df2)
display(df_union)

# DATE FUNCTIONS

In [ ]:
df_curr = df_csv.withColumn("current_time",current_timestamp())
display(df_curr)

In [ ]:
df_add = df_curr.withColumn("current_time",date_add('current_time',7))
display(df_add)

In [ ]:
df_sub = df_curr.withColumn("current_time",date_sub('current_time',7))
display(df_sub)

In [ ]:
df_diff = df_curr.withColumn("duration",date_diff('current_time','order_date'))
display(df_diff)

In [ ]:
df_format = df_curr.withColumn("current_time",date_format('current_time','yyyy-MM-dd'))
display(df_format)

# STRING FUNCTIONS

In [ ]:
df_str = df_csv.withColumn("order_status",lower('order_status'))
display(df_str)

In [ ]:
df_str = df_csv.withColumn("order_status_length",length('order_status'))
display(df_str)

# Handling Nulls

In [ ]:
data = [
    (1, 'abc', None),
    (2, 'def', 'xyz'),
    (3, 'ghi', 'xyz'),
    (4, 'jkl', 'zyx'),
    (None, None, None),
    (6, 'pqr', 'bbb'),
    (7, 'stu', 'ihd'),
    (8, 'vwx', None),
    (9, 'yz', None)
]

df = spark.createDataFrame(
    data,
    ['id', 'name', 'address']
)
display(df)

### All Nulls

In [ ]:
df_all_nulls = df.dropna('all')
display(df_all_nulls)

### Any Null

In [ ]:
df_any_null = df.dropna('any')
display(df_any_null)

### Subset

In [ ]:
df_subset_null = df.dropna(subset=['name','address'],how='all')
display(df_subset_null)

### Fill Nulls

In [ ]:
df_fillna = df.fillna('dummy')
display(df_fillna)

In [ ]:
df_custom_fillna = df.fillna({'name':'dummy_name','address':'dummy_address'})
display(df_custom_fillna)

# Split & Indexing

In [ ]:
df_split = df_csv.withColumn("street_address",split('shipping_address',',')[0])\
                .withColumn("city_name",split('shipping_address',',')[1])

display(df_split)

# EXPLODE

In [ ]:
df_arr = df_csv.withColumn("address_array",split("shipping_address",","))\
                .select("order_id","address_array")       
display(df_arr)

In [ ]:
df_exp = df_arr.withColumn("address_explode",explode("address_array"))
display(df_exp)

In [ ]:
df_exp = df_arr.withColumn("address_explode",explode_outer("address_array"))
display(df_exp)

# Array Contains

In [ ]:
df_arr_con = df_arr.withColumn("city_1_flag",array_contains("address_array"," City1"))
display(df_arr_con)

# Group By

In [ ]:
df_agg = df_csv.withColumn("total_price",col('price')*col('quantity'))\
            .groupBy('product_id').agg(sum('total_price').alias('total_price'))\
            .withColumn("total_price",round(col('total_price'),2))\
            .sort(col('total_price').desc())


display(df_agg)

In [ ]:
df_agg_new = df_csv.withColumn("total_price",col('price')*col('quantity'))\
            .groupBy('product_id','customer_id').agg(sum('total_price').alias('total_price'),avg('total_price').alias('avg_price'))\
            .withColumn("total_price",round(col('total_price'),2))\
            .withColumn("avg_price",round(col('avg_price'),2))\
            .sort('product_id','total_price',ascending=[True,False])

display(df_agg_new)
            

# **Approx Count**

In [ ]:
df = df_csv.groupBy('product_id').agg(approx_count_distinct('customer_id').alias('distinct'))
display(df)

# Collect List

In [ ]:
df_collect = df_csv.groupBy('customer_id').agg(collect_list('product_id').alias('products'))
display(df_collect)

# PIVOT

In [ ]:
df_pivot = df_csv.groupBy('customer_id').pivot('order_status').agg(count('order_id'))
display(df_pivot)

# When-Otherwise

In [ ]:
df_ifelse = df_pivot.withColumn('return_flag',when(col('Returned')==1,'low').when(col('Returned')<=3,'mid').when(col('Returned')>3,'high').otherwise('no returns'))

display(df_ifelse)

In [ ]:
df_ifelse_new = df_pivot.withColumn('return_flag',when(col('Returned')==1,'low').when(((col('Returned')==3) & (col('Returned')==2)) ,'mid').when(col('Returned')>3,'high').otherwise('no returns'))

display(df_ifelse_new)

# JOINS

In [ ]:
data1 = [(1,'abc',901),(2,'def',902),(3,'ghi',903),(4,'jkl',904),(5,'mno',905),(6,'pqr',906),(7,'stu',907),(8,'vwx',908),(9,'tuv',909),(10,'wxy',910),(11,'zab',911),(12,'bcd',912),(13,'efg',913),(14,'hij',914),(15,'klm',915),(16,'nop',916),(17,'qrs',917),(18,'tuv',918),(19,'wxy',919),(20,'zab',920),(21,'bcd',921),(22,'efg',922),(23,'hij',923),(24,'klm',924),(25,'nop',925)]

df1 = spark.createDataFrame(data1,['id','name','code'])

data2 = [(101,1),(102,2),(103,3),(104,4),(105,5),(106,6),(107,7),(108,8),(109,9),(110,10),(111,11),(112,12),(113,13),(114,14),(115,15),(116,31),(117,32),(118,33),(119,34),(120,35),(121,36),(122,37),(123,38),(124,39),(125,40)]
df2 = spark.createDataFrame(data2,['o_id','id'])

display(df1)
display(df2)

### Inner Join

In [ ]:
df_inner = df1.join(df2,df1['id']==df2['id'],'inner')
display(df_inner)

### Left Join

In [ ]:
df_inner = df1.join(df2,df1['id']==df2['id'],'left')
display(df_left)

### Right Join

In [ ]:
df_right = df1.join(df2,df1['id']==df2['id'],'right')
display(df_right)

In [ ]:
df_right = df2.join(df1,df1['id']==df2['id'],'left')
display(df_inner)

### Full Join

In [ ]:
df_full = df1.join(df2,df1['id']==df2['id'],'full')
display(df_full)

### Anti Join

In [ ]:
df_anti = df1.join(df2,df1['id']==df2['id'],'anti')
display(df_anti)

# WINDOW FUNCTIONS

In [ ]:
from pyspark.sql.window import Window

In [ ]:
data = [(33),(45),(67),(99),(99),(100)]

df = spark.createDataFrame(data,"marks INT")
display(df)

In [ ]:
df_row = df.withColumn("marks_sequence",row_number().over(Window.orderBy('marks')))
display(df_row)

In [ ]:
df_row = df.withColumn("marks_sequence",row_number().over(Window.partitionBy('marks').orderBy('marks')))
display(df_row)

### Rank VS Dense_Rank

In [ ]:
df_ranking = df.withColumn("rank",rank().over(Window.orderBy('marks')))\
                .withColumn("dense_rank",dense_rank().over(Window.orderBy('marks')))

display(df_ranking)

### Total Revenue

In [ ]:
data = [(2020,100),(2021,110),(2022,140),(2023,90),(2024,150)]

df = spark.createDataFrame(data,"year INT,revenue INT")
display(df)

In [ ]:
df_total = df.withColumn("total_revenue",sum('revenue').over(Window.orderBy('year')))

display(df_total)

In [ ]:
df_total = df.withColumn("total_revenue",sum('revenue').over(Window.orderBy('year').rowsBetween(Window.unboundedPreceding,Window.currentRow)))

display(df_total)

In [ ]:
df_total = df.withColumn("total_revenue",sum('revenue').over(Window.orderBy('year').rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing)))

display(df_total)

# User Defined Functions

In [ ]:
def my_square(x):
    square = x*x
    return square

In [ ]:
my_square_udf = udf(my_square,FloatType())

In [ ]:
df_cust = df_csv.withColumn("price_sqaure",my_square_udf('price'))
display(df_cust)

In [ ]:
@udf(returnType=FloatType())
def my_square_new(x):
    square = x*x
    return square

In [ ]:
df_cust = df_csv.withColumn("price_sqaure",my_square_new('price'))
display(df_cust)

# User Defined Table Functions (UDTF)

In [ ]:
df_demo = spark.createDataFrame([("Hello World",), ("Just an example",), ("Hi Bro",)], ["text"])
display(df_demo)

In [ ]:
@udtf(returnType="word STRING")
class udtf_class:
    def eval(self, text: str): # Function name to be used
        for word in text.split():
            yield (word,)

In [ ]:
udtf_class(lit("Hello world, this is a developer")).show()

# Call UDF

In [ ]:
spark.udf.register("my_square_new", my_square_new)

df_cust_new = df_csv.withColumn("price_sqaure_new",call_udf("my_square_new",col('price')))
display(df_cust_new)

# Concat & ConcatWS

### Concat

In [ ]:
df_con = df_csv.withColumn("custom_id",concat(col('customer_id'),lit('|'),col('order_id'),lit('|'),col('product_id')))

display(df_con)

In [ ]:
df_con = df_csv.withColumn("custom_id",concat_ws(lit('|'),col('customer_id'),col('order_id'),col('product_id')))

display(df_con)

# DATA WRITING

### Overwrite

In [ ]:
df_csv.write.format("csv")\
        .mode("overwrite")\
        .option("path","/Volumes/sparkcatalog/raw/source/destination/csv_overwrite")\
        .save()

### Append

In [ ]:
df_csv.write.format("csv")\
        .mode("append")\
        .option("path","/Volumes/sparkcatalog/raw/source/destination/csv_overwrite")\
        .save()

### Error

In [ ]:
# df_csv.write.format("csv")\
#         .mode("error")\
#         .option("path","/Volumes/sparkcatalog/raw/source/destination/csv_overwrite")\
#         .save()

### Ignore

In [ ]:
df_csv.write.format("csv")\
        .mode("ignore")\
        .option("path","/Volumes/sparkcatalog/raw/source/destination/csv_overwrite")\
        .save()

# File Formats

### CSV

In [ ]:
df_csv.write.format("csv")\
        .mode("append")\
        .option("path","/Volumes/sparkcatalog/raw/source/destination/csv")\
        .save()

### JSON

In [ ]:
df_csv.write.format("json")\
        .mode("append")\
        .option("path","/Volumes/sparkcatalog/raw/source/destination/json")\
        .save()

### Parquet

In [ ]:
df_csv.write.format("parquet")\
        .mode("append")\
        .option("path","/Volumes/sparkcatalog/raw/source/destination/parquet")\
        .save()

### Delta

In [ ]:
df_csv.write.format("delta")\
        .mode("append")\
        .option("path","/Volumes/sparkcatalog/raw/source/destination/delta")\
        .save()

# UPSERT With Delta Library

In [ ]:
df_new = df_csv.filter(col('order_id').isin(1001,1002))
df_new_1 = df_new.filter(col('order_id')==1001).withColumn("product_id",lit("P102"))
df_new_2 = df_new.filter(col('order_id')==1002).withColumn("order_id",lit("90001"))

df_new = df_new_1.union(df_new_2)
display(df_new)

In [ ]:
from delta.tables import DeltaTable

dlt_obj = DeltaTable.forPath(spark,"/Volumes/sparkcatalog/raw/source/destination/delta/")

dlt_obj.alias("trg").merge(df_new.alias("src"),"trg.order_id = src.order_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

In [ ]:
df_test = spark.read.format("delta")\
            .load("/Volumes/sparkcatalog/raw/source/destination/delta")
display(df_test)

# Handling JSON Data

In [ ]:
df_json = spark.read.format("json")\
                .option("inferSchema",True)\
                .option("multiLine",True)\
                .load("/Volumes/sparkcatalog/raw/source/json_orders/")

# Explode items
df_json = df_json.withColumn("items",explode(col("items")))

# Fetching cols 
df_json = df_json.withColumn("item_id",col("items.item_id"))\
                 .withColumn("quantity",col("items.quantity"))\
                  .withColumn("price",col("items.price"))\
                .withColumn("product_name",col("items.product_name"))\
                .drop("items")\
                .withColumn("customer_id",col("customer.customer_id"))\
                .withColumn("email",col("customer.email"))\
                .withColumn("name",col("customer.name"))\
                .withColumn("city",col("customer.address.city"))\
                .drop("customer")\
                .withColumn("method",col("payment.method"))\
                .withColumn("transaction_id",col("payment.transaction_id"))\
                .drop("payment","metadata")
                


display(df_json)

# SparkSQL 

### Temporary View

In [ ]:
df_csv.createOrReplaceTempView("csv_view")

In [ ]:
display(spark.sql("SELECT * FROM csv_view"))

In [ ]:
df_sql = spark.sql("""
          SELECT * FROM csv_view
          WHERE order_id IN (1001,1002,90001)""")

In [ ]:
display(df_sql)

### Global Temp View

In [ ]:
df_csv.createOrReplaceGlobalTempView("csv_view_global")

### DDL

In [ ]:
spark.sql("CREATE TABLE IF NOT EXISTS sparkcatalog.raw.new_table")

In [ ]:
spark.sql("""
          CREATE TABLE sparkcatalog.raw.schema_table
          (
                ID INT,
                NAME STRING,
                AGE INT
          )
          """)

In [ ]:
spark.sql(
    """
    INSERT INTO sparkcatalog.raw.schema_table
    VALUES
    (1, 'John', 25),
    (2, 'Jane', 30),
    (3, 'Bob', 35)
    """
)

In [ ]:
display(spark.sql("SELECT * FROM sparkcatalog.raw.schema_table"))

### JOINS

In [ ]:
spark.sql("""
          CREATE TABLE sparkcatalog.raw.schema_table_2
          (
                ID INT,
                ADDRESS STRING
          )
          """)

spark.sql("""
            INSERT INTO sparkcatalog.raw.schema_table_2
            VALUES
            (1, '123 Main St'),
            (2, '456 Oak Ave'),
            (3, '789 Pine Ln')
          """)

In [ ]:
display(spark.sql("""
        SELECT * 
        FROM 
            sparkcatalog.raw.schema_table 
            LEFT JOIN 
                sparkcatalog.raw.schema_table_2
                ON 
                    schema_table.ID = schema_table_2.ID
          """))

### UPSERT

In [ ]:
data = [(1, 'John', 25), (2, 'Jane', 33),(4, 'Bobi', 35)]
df_new = spark.createDataFrame(data,"id int, name string, age int")

display(df_new)

In [ ]:
df_new.createOrReplaceTempView("new_data")

# Applying Upsert With Merge Command
spark.sql("""
          MERGE INTO sparkcatalog.raw.schema_table t
          USING new_data s
          ON t.ID = s.ID
          WHEN MATCHED THEN
            UPDATE SET *
            WHEN NOT MATCHED
            THEN INSERT *
          """)
display(spark.sql("SELECT * FROM sparkcatalog.raw.schema_table"))

### MergeInto API (Latest Spark 4.0+)

In [ ]:
data = [(1, 'John', 25), (2, 'Jane', 36),(4, 'Rob', 35)]
df_new_2 = spark.createDataFrame(data,"id int, name string, age int")

display(df_new_2)

In [ ]:
df_new.alias('src').mergeInto("sparkcatalog.raw.schema_table", col("src.ID") == col("sparkcatalog.raw.schema_table.ID"))\
        .whenMatched().updateAll()\
        .whenNotMatched().insertAll()\
        .merge()

In [ ]:
display(spark.sql("SELECT * FROM sparkcatalog.raw.schema_table"))

### Partition BY

In [ ]:
%sql
CREATE TABLE sparkcatalog.raw.part_table
(
  id INT,
  name STRING
)
USING delta
PARTITIONED BY (id)

In [ ]:
%sql
INSERT INTO sparkcatalog.raw.part_table
VALUES (1, 'a'), (2, 'b'), (3, 'c');

In [ ]:
%sql
ALTER TABLE sparkcatalog.raw.part_table;

DROP TABLE sparkcatalog.raw.part_table;

### EXPLAIN

In [ ]:
%sql
EXPLAIN SELECT * FROM sparkcatalog.raw.part_table;

### SQL Functions
You can use all of the SQL functions as-it-is here in SparkSQL as well

In [ ]:
%sql
SELECT * FROM sparkcatalog.raw.part_table

In [ ]:
%sql 

WITH inner_query AS (
  SELECT 
    *,
    CASE 
    WHEN mod(id, 2) = 0 THEN 'even'
    ELSE 'odd'
    END AS even_odd_flag
  FROM
    sparkcatalog.raw.part_table 
  )
SELECT * FROM inner_query

# SQL Scripting Statements

### For

In [ ]:
%sql
BEGIN
    DECLARE sum INT DEFAULT 0;
    sumNumbers: FOR row AS SELECT num FROM range(1, 20) AS t(num) DO
      IF num > 10 THEN
         LEAVE sumNumbers;
      ELSEIF num % 2 = 0 THEN
        ITERATE sumNumbers;
      END IF;
      SET sum = sum + row.num;
    END FOR sumNumbers;
    VALUES (sum);
  END;

### If

In [ ]:
%sql
BEGIN
    DECLARE choice DOUBLE DEFAULT 3.9;
    DECLARE result STRING;
    IF choice < 2 THEN
      VALUES ('one fish');
    ELSEIF choice < 3 THEN
      VALUES ('two fish');
    ELSEIF choice < 4 THEN
      VALUES ('red fish');
    ELSEIF choice < 5 OR choice IS NULL THEN
      VALUES ('blue fish');
    ELSE
      VALUES ('no fish');
    END IF;
  END;

### Case

In [ ]:
%sql
BEGIN
    DECLARE choice INT DEFAULT 3;
    DECLARE result STRING;
    CASE choice
      WHEN 1 THEN
        VALUES ('one fish');
      WHEN 2 THEN
        VALUES ('two fish');
      WHEN 3 THEN
        VALUES ('red fish');
      WHEN 4 THEN
        VALUES ('blue fish');
      ELSE
        VALUES ('no fish');
    END CASE;
  END;

### While

In [ ]:
%sql
BEGIN
    DECLARE sum INT DEFAULT 0;
    DECLARE num INT DEFAULT 0;
    sumNumbers: WHILE num < 10 DO
      SET num = num + 1;
      IF num % 2 = 0 THEN
        ITERATE sumNumbers;
      END IF;
      SET sum = sum + num;
    END WHILE sumNumbers;
    VALUES (sum);
  END;

# SPARKSQL Auxiliary Statements

### DESCRIBE

In [ ]:
%sql
DESCRIBE DATABASE sparkcatalog.raw

In [ ]:
%sql
DESCRIBE TABLE sparkcatalog.raw.part_table

In [ ]:
%sql 
DESCRIBE QUERY SELECT * FROM sparkcatalog.raw.part_table

### REFRESH 

In [ ]:
%sql
REFRESH TABLE sparkcatalog.raw.part_table

### SHOW

In [ ]:
%sql
SHOW SCHEMAS FROM sparkcatalog

In [ ]:
%sql
SHOW TABLES FROM sparkcatalog.raw

In [ ]:
%sql 
USE sparkcatalog.raw;
SHOW TABLE EXTENDED LIKE 'part_table'

In [ ]:
%sql 
SHOW TBLPROPERTIES sparkcatalog.raw.part_table

In [ ]:
%sql 
SHOW PARTITIONS sparkcatalog.raw.part_table

# SPARKSQL Advanced Functions

### Aggregate Functions

In [ ]:
df_csv.createOrReplaceTempView("sql_tbl")

In [ ]:
%sql
SELECT * FROM sql_tbl

In [ ]:
# pySpark Way
display(df_csv.groupBy("customer_id").agg(array_agg("order_id")))

In [ ]:
%sql
SELECT customer_id, array_agg(order_id) FROM sql_tbl GROUP BY customer_id

In [ ]:
%sql
SELECT 
  customer_id,
  collect_list(product_id),
  collect_set(product_id) 
FROM sql_tbl
GROUP BY customer_id

In [ ]:
%sql
SELECT 
    corr(price,quantity),
    max(price),
    avg(price),
    min(price),
    median(price),
    mode(price)
FROM 
  sql_tbl

### STRUCT & MAP

In [ ]:
%sql
SELECT struct(1,2,3,"abc")

In [ ]:
%sql
SELECT  named_struct("a",1,"b",2,"c",3)

In [ ]:
%sql
SELECT  map("a",1,"b",2,"c",3)

In [ ]:
%sql
SELECT map_values(map("a", 1, "b", 2, "c", 3));


SELECT map_keys(map("a", 1, "b", 2, "c", 3));

### DATETIME FUNCTIONS

In [ ]:
%sql
SELECT current_timestamp();

SELECT add_months(current_timestamp(), 1);

SELECT add_months(current_timestamp(), -1);

SELECT convert_timezone('America/Los_Angeles', current_timestamp());

In [ ]:
%sql
SELECT unix_timestamp()

In [ ]:
%sql
SELECT date_from_unix_date(0);

SELECT from_unixtime(0);

SELECT from_utc_timestamp(current_timestamp(), 'America/Los_Angeles')

In [ ]:
%sql
SELECT dayname(current_timestamp()),
      dayofmonth(current_timestamp()),
      dayofweek(current_timestamp())

In [ ]:
%sql
SELECT 
    extract(year from current_timestamp()),
    extract(month from current_timestamp()),
    extract(day from current_timestamp()),
    extract(hour from current_timestamp()),
    extract(minute from current_timestamp()),
    extract(second from current_timestamp()),
    extract(doy from current_timestamp())

### Window Functions

In [ ]:
%sql
SELECT 
  order_id,
  LAG(order_date, 1, '1900-01-01') OVER(order by order_id) AS prev_order_date,
  order_date,
  LEAD(order_date, 1, '9999-12-31') OVER(order by order_id) AS next_order_date
FROM 
  sql_tbl;

In [ ]:
%sql
SELECT 
    order_id,
    customer_id,
    order_status,
    ntile(3) OVER(PARTITION BY order_status ORDER BY order_id) AS order_group
FROM
  sql_tbl

### Array Functions

In [ ]:
%sql

SELECT array(1,2,3);

SELECT array_append(array(1,2,3),9);

SELECT array_distinct(array(1,2,3,3));

SELECT array_compact(array(1,2,NULL,3,NULL,3));

SELECT array_contains(array(1,2,3),9);

SELECT array_insert(array(1,2,3), 2, 9);

SELECT array_prepend(array(1,2,3), 0);

SELECT array_position(array(1,3,3,4,2), 2);

# UDFs In SparkSQL

In [ ]:
def my_upper_function(input:str)->str:
    return input.upper()

In [ ]:
spark.udf.register("my_upper", my_upper_function, StringType())

In [ ]:
%sql
SELECT 
  my_upper('hello')

# SparkSQL Using DF

In [ ]:
display(spark.sql("SELECT * FROM {df_csv_var} WHERE price > 10 ", df_csv_var = df_csv))

# SPARKSQL Query Files

### 1) Using Views On Top Of Files

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_csv
USING CSV
OPTIONS(
  path "/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv",
  header "true",
  inferSchema "true"
);

SELECT * FROM orders_csv;

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW orders_json
USING org.apache.spark.sql.json
OPTIONS(
  path "/Volumes/sparkcatalog/raw/source/raw_orders/orders.json",
  header "true",
  inferSchema "true"
);

SELECT * FROM orders_json;

### 2) Using File Format Connector

In [ ]:
display(spark.sql("SELECT * FROM json.`/Volumes/sparkcatalog/raw/source/raw_orders/orders.json`"))

# Broadcast Hash Join

In [ ]:
data1 = [(1,'abc',901),(2,'def',902),(3,'ghi',903),(4,'jkl',904),(5,'mno',905),(6,'pqr',906),(7,'stu',907),(8,'vwx',908),(9,'tuv',909),(10,'wxy',910),(11,'zab',911),(12,'bcd',912),(13,'efg',913),(14,'hij',914),(15,'klm',915),(16,'nop',916),(17,'qrs',917),(18,'tuv',918),(19,'wxy',919),(20,'zab',920),(21,'bcd',921),(22,'efg',922),(23,'hij',923),(24,'klm',924),(25,'nop',925)]

df1 = spark.createDataFrame(data1,['id','name','code'])

data2 = [(101,1),(102,2),(103,3),(104,4),(105,5),(106,6),(107,7),(108,8),(109,9),(110,10),(111,11),(112,12),(113,13),(114,14),(115,15),(116,31),(117,32),(118,33),(119,34),(120,35),(121,36),(122,37),(123,38),(124,39),(125,40)]
df2 = spark.createDataFrame(data2,['o_id','id'])

# Turn OFF the AQE
spark.conf.set("spark.sql.adaptive.enabled","false")

# Broadcast Join
df_broadcast = df1.join(broadcast(df2),df1['id']==df2['id'],'inner')
display(df_broadcast)

# Cache & Persist

In [ ]:
df_csv.cache()

In [ ]:
display(df_csv)

In [ ]:
from pyspark.storagelevel import StorageLevel

df_json.persist(StorageLevel.DISK_ONLY)

In [ ]:
display(df_json)

In [ ]:
df_csv.unpersist()
df_json.unpersist()

In [ ]:
display(df_json)

# Partitions in PySpark

In [ ]:
df_part = spark.read.format("csv")\
      .option("header",True)\
      .option("inferSchema",True)\
      .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

display(df_part)

In [ ]:
df_part = df_part.withColumn("year",year(col("order_date")))\
                .withColumn("month",month(col("order_date")))\
                .withColumn("day",dayofmonth(col("order_date")))

display(df_part)

In [ ]:
df_part.write.format("parquet")\
            .mode("append")\
            .partitionBy("year","month","day")\
            .option("path","/Volumes/sparkcatalog/rawsource/orders_partitions/")\
            .save()

# Partition Pruning

In [ ]:
df_prune = spark.read.format("parquet")\
              .load("/Volumes/sparkcatalog/raw/source/orders_partitions/")\
              .filter((col("year") == 2025) & (col("month")==12) & (col("day")==25))

display(df_prune)

# SALTING WITH GroupBY

In [ ]:
data = [(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(1,'food'),(2,'electronics'),(2,'electronics'),(2,'electronics'),(2,'electronics'),(3,'clothes'),(3,'clothes'),(3,'clothes'),(3,'clothes'),(3,'clothes')]

df_skew = spark.createDataFrame(data,['id','category'])

display(df_skew)

In [ ]:
df_salt = df_skew.withColumn("salt_col",floor(rand()*3))
display(df_salt)

In [ ]:
df_salt = df_salt.withColumn("salt_group",concat(col('id'),lit('-'),col('salt_col')))
display(df_salt)

In [ ]:
df_grp = df_salt.groupBy("salt_group").agg(count("category").alias("total_count"))
display(df_grp)

# SALTING WITH JOINS

In [ ]:
data = [(1,'edible'),(2,'lifestyle'),(3,'nature')]

df_small = spark.createDataFrame(data,["id","tag"])

# Adding Salat Array
df_small = df_small.withColumn("salt_array",array([lit(i) for i in range(3)]))

# Explode the salt_array
df_small = df_small.select("id","tag",explode("salt_array").alias("salt_col"))
display(df_small)

In [ ]:
display(df_salt)

In [ ]:
df_join = df_salt.join(df_small,(df_salt['id']==df_small['id'])&(df_salt['salt_col']==df_small['salt_col']),"left")
display(df_join)

# SQL Hints

In [ ]:
# Hypothetical Example
df1 = df_salt
df2 = df_small

df1.createOrReplaceTempView("df1")
df2.createOrReplaceTempView("df2")


# SQL Hints
display(spark.sql("""
SELECT /*+ BROADCAST(df2) */ 
* 
FROM 
    df1 
JOIN df2 
ON df1.id = df2.id AND df1.salt_col = df2.salt_col"""))

# Broadcast Variable

In [ ]:
my_map = {
    "1" : "edible",
    "2" : "lifestyle",
    "3" : "technology"
}

In [ ]:
spark.sparkContext.broadcast(my_map)

In [ ]:
map_value = spark.sparkContext.broadcast(my_map)
map_value.value

In [ ]:
map_value.value['1']

# 8 Spark Structured Streaming

*Notebook 8 of 9*

In [ ]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

# Append Mode

In [ ]:
# Enabling Schema Inference 
spark.conf.set("spark.sql.streaming.streaming.schemaInference" ,"True")

In [ ]:
df_batch = spark.read.format("json")\
        .option("multiLine",True)\
        .option("inferSchema",True)\
        .load("/Volumes/sparkcatalog/raw/source/src_append/")

json_schema = df_batch.schema

In [ ]:
df = spark.readStream.format("json")\
        .option("multiLine",True)\
        .schema(json_schema)\
        .option("cleanSource","archive")\
        .option("sourceArchiveDir","/Volumes/sparkcatalog/raw/source/source_archive/")\
        .load("/Volumes/sparkcatalog/raw/source/src_append/")

In [ ]:
query = df.writeStream.format("delta")\
        .outputMode("append")\
        .trigger(processingTime="5 seconds")\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_append/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_append/data")\
        .start()

In [ ]:
query.stop()

In [ ]:
df_res = spark.read.format("delta")\
            .load("/Volumes/sparkcatalog/raw/source/sink_append/data")

display(df_res)

# Complete Mode

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS sparkcatalog.raw.complete_source
(
  color STRING
)
USING DELTA

In [ ]:
%sql
INSERT INTO sparkcatalog.raw.complete_source
VALUES ('red')

In [ ]:
df = spark.readStream.table('sparkcatalog.raw.complete_source')

df = df.groupBy('color').agg(count('color').alias('count'))

df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(processingTime="5 seconds")\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()

In [ ]:
df_sink = spark.read.format("delta")\
            .load("/Volumes/sparkcatalog/raw/source/sink_complete/data")

display(df_sink)

# Trigger Modes

In [ ]:
# Processing Time
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(processingTime="5 seconds")\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()
    
# Once 
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(once=True)\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()

# Available Now
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(availableNow=True)\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()

# Continuous (Not Supported Yet With Delta Tables)
df.writeStream.format("delta")\
        .outputMode("complete")\
        .trigger(continuous='1 second')\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/sink_complete/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/sink_complete/data")\
        .start()

# ForEachBatch

In [ ]:
df = spark.readStream.table('sparkcatalog.raw.complete_source')

In [ ]:
def my_function(df,batch_id):

    # Writing To Stream 1 (Will Have To Use Batch Code)
    df.write.format("parquet")\
            .mode("overwrite")\
            .option("path","/Volumes/sparkcatalog/raw/source/streams/stream1")\
            .save()

    # Writing To Stream 2 (Will Have To Use Batch Code)
    df.write.format("parquet")\
            .mode("overwrite")\
            .option("path","/Volumes/sparkcatalog/raw/source/streams/stream2")\
            .save()

    # Can Use Batch Logic To Apply UPSERT 

In [ ]:
df.writeStream.foreachBatch(my_function)\
        .outputMode("append")\
        .option("path","/Volumes/sparkcatalog/raw/source/streams/checkpoint")\
        .option("path","/Volumes/sparkcatalog/raw/source/streams/data")\
        .start()
        
        

# WINDOWS

In [ ]:
%sql 
-- Source Table

CREATE TABLE sparkcatalog.raw.stream_source
(
  color STRING,
  event_time TIMESTAMP
)

In [ ]:
%sql
select current_timestamp()

In [ ]:
%sql
INSERT INTO sparkcatalog.raw.stream_source
VALUES
('GREEN','2026-02-12T20:08:18.360+00:00')

In [ ]:
df_str = spark.readStream.table('sparkcatalog.raw.stream_source')\
                .groupBy('color',window('event_time','10 minutes')).agg(count('color').alias("count"))

In [ ]:
df_str.writeStream.format("delta")\
        .outputMode("complete")\
        .option("checkpointLocation", "/Volumes/sparkcatalog/raw/source/window_sink/checkpoint")\
        .option("path", "/Volumes/sparkcatalog/raw/source/window_sink/data")\
        .start()

In [ ]:
df_res = spark.read.format("delta").load("/Volumes/sparkcatalog/raw/source/window_sink/data")
display(df_res)

### WATERMARKS

In [ ]:
df_str = spark.readStream.table('sparkcatalog.raw.stream_source')\
                .withWatermark('event_time','10 minutes')\
                .groupBy('color',window('event_time','10 minutes')).agg(count('color').alias("count"))

# 9 PandasAPI

*Notebook 9 of 9*

In [ ]:
import pyspark.pandas as ps
import pandas as pd

In [ ]:
df = ps.read_csv("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df[['order_id','customer_id','price']].head()

In [ ]:
df['new_col'] = df['price']*10
df.head()

In [ ]:
df['flag'] = df['price']>1000
df.head()

In [ ]:
df[df['price']>10].head()

In [ ]:
df.sort_values('price',ascending=False).head()

# Pandas UDF

### Series To Series

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [ ]:
@pandas_udf("double")
def price_func(col1:pd.Series,col2:pd.Series) -> pd.Series:
    return col1 * col2

In [ ]:
# Creating a PySpark Dataframe
df_spark = spark.read.format("csv")\
                .option("header","true")\
                .option("inferSchema","true")\
                .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")


df_spark = df_spark.withColumn("total_price",price_func('price','quantity'))

display(df_spark)

### Iterator Series To Iterator Series

In [ ]:
from typing import Iterator, Tuple

@pandas_udf("double")
def iter_func(col1: Iterator[pd.Series]) -> Iterator[pd.Series]:

    for batch in col1:
        yield batch * 10

  

In [ ]:
df_spark = df_spark.withColumn("iter_price",iter_func('price'))
display(df_spark)

%md
### Multiple Iterator Series To Iterator Series

In [ ]:
@pandas_udf("double")
def price_iter_mul(cols: Iterator[Tuple[pd.Series, pd.Series]]) -> Iterator[pd.Series]:
    
    for col1, col2 in cols:
        yield col1 * col2

In [ ]:
df_spark = df_spark.withColumn("iter_price_mul",price_iter_mul('price','quantity'))
display(df_spark)

%md
### Series To Scalar

In [ ]:
@pandas_udf("double")
def scalar_func(col1: pd.Series) -> float:
  return col1.mean()
  

In [ ]:
df_spark = df_spark.groupBy("customer_id").agg(scalar_func('price').alias("mean_price"))
display(df_spark)

### Using It With Spark SQL

In [ ]:
df_spark = spark.read.format("csv")\
                .option("header","true")\
                .option("inferSchema","true")\
                .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

In [ ]:
df_spark.createOrReplaceTempView("df_spark")

In [ ]:
spark.udf.register("price_iter_mul_udf", price_iter_mul)

In [ ]:
display(spark.sql("""
          SELECT 
            *,
            price_iter_mul_udf(price, quantity) as iter_price_mul
        FROM 
            df_spark"""))